In [ ]:
!pip install zstandard python-chess
import zstandard as zstd
import chess.pgn
import io

zst_path = "/content/drive/MyDrive/lichess_db_standard_rated_2015-03.pgn.zst"

with open(zst_path, 'rb') as compressed:
    dctx = zstd.ZstdDecompressor()
    stream_reader = dctx.stream_reader(compressed)
    text_stream = io.TextIOWrapper(stream_reader, encoding = 'utf-8')


    first_game = chess.pgn.read_game(text_stream)


print("=== Game Headers ===")
for key, value in first_game.headers.items():
    print(f"{key}: {value}")


print("\n=== Moves ===")
board = first_game.board()
move_list = []
for move in first_game.mainline_moves():
    san = board.san(move)
    move_list.append(san)
    board.push(move)


for i in range(0, len(move_list), 2):
    move_num = i // 2 + 1
    white_move = move_list[i]
    black_move = move_list[i + 1] if i + 1 < len(move_list) else ""
    print(f"{move_num}. {white_move} {black_move}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 40.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=a8eadf59971aa5ec8d6d7bff8e57f796362ab605f881bbe858b05f9fafc32909
  Stored in directory: /root/.cache/pip/wheels/fb/5d/5c/59a62d8a695285e59ec9c1f66add6f8a9ac4152499a2be0113
Successfully built chess
=== Game Headers ===
Event: Rated Blitz game
Site: https://lichess.org/f2Oejy7P
Date: ????.??.??
Round: ?
White: inthisposition
Black: Boletov
Result: 1-0
UTCDate: 2015.02.28
UTCTime: 23:00:00
WhiteElo: 1821
BlackElo: 1785
WhiteRatingDiff: +10
BlackRatingDiff: -17
ECO: A85
Opening: Dutch Defense: Queen's Knight Variation
TimeControl: 300+0
Termination: Time forfeit
LichessId: f2Oejy7P

=== Moves ===
1. d4 f5
2. c4 Nf6
3. Nc3 g6
4. g3 Bg7
5. Bg2 O-O
6. Nf3 d6
7. O-O Qe8
8. d5 a5
9. a4 Na6
10. Re1 Nc5
11. Ng5 h6
12. Nf3 Qf7
13. Nb5 e5
14. dxe6 Bxe6
15. Nfd4 Bxc4
16. Ra3 Nfe4
17. b3 Bd5
18. f3 

In [ ]:
import chess
import zstandard as zstd
import io
import chess.pgn
import csv

pgn_path = "/content/drive/MyDrive/lichess_db_standard_rated_2015-03.pgn.zst"

def extract_game_data(game):
    if game is None:
        return None


    headers = game.headers
    white_elo = headers.get('WhiteElo', 'N/A')
    black_elo = headers.get('BlackElo', 'N/A')
    result = headers.get('Result', 'N/A')


    board = chess.Board()


    moves = []
    for move in game.mainline_moves():
        san_move = board.san(move)
        moves.append(san_move)
        board.push(move)

    return white_elo, black_elo, result, moves


with open("games_with_fens.csv", "w", encoding="utf-8", newline='') as out_csv:
    csv_writer = csv.writer(out_csv)
    csv_writer.writerow(["game_id", "white_elo", "black_elo", "result", "moves"])


    with open(pgn_path, "rb") as compressed:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed) as reader:
            text_stream = io.TextIOWrapper(reader, encoding = "utf-8")
            game_id = 1

            while True:
                game = chess.pgn.read_game(text_stream)
                if not game:
                    break

                data = extract_game_data(game)
                if data:
                    white_elo, black_elo, result, moves = data
                    moves_str = ' '.join(moves)
                    csv_writer.writerow([game_id, white_elo, black_elo, result, moves_str])
                    game_id += 1

In [ ]:
import pandas as pd
df = pd.read_csv('/content/games_with_fens.csv')
df.head()


<ipython-input-15-ac6e75284af4>:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/games_with_fens.csv')


,game_id,white_elo,black_elo,result,moves
0,1,1821,1785,1-0,d4 f5 c4 Nf6 Nc3 g6 g3 Bg7 Bg2 O-O Nf3 d6 O-O ...
1,2,1302,1655,0-1,e4 e5 Nf3 Nc6 Bb5 Nf6 d3 a6 Bxc6 dxc6 Nxe5 Bg4...
2,3,1500,1636,0-1,d3 Nf6 e3 e5 d4 exd4 exd4 d5 Nf3 Bd6 c4 O-O
3,4,1767,1700,1-0,d4 g6 c4 Bg7 Nc3 e6 e4 a6 Nf3 Ne7 Bg5 Nbc6 Qd2...
4,5,1741,1409,0-1,e4 c5 Bc4 e6 a3 a6 d3 d5 exd5 exd5 Ba2 Nf6 h3 ...


In [ ]:
len(df)

1742733